**COMMAND:** `nice` / `renice` — CPU priority control

**PURPOSE (support):** Sometimes a process isn't broken, it's just hogging CPU and starving everything else — a backup script, a log-compression job, a data migration. Killing it isn't the answer; you want it to run _slower relative to other processes_ so the box stays responsive. This is priority management, not process control.

**SYNTAX:**

- `nice -n VALUE command` — launch a new process with adjusted priority
- `renice -n VALUE -p PID` — change priority of an already-running process

**FLAGS:**

|Flag|Meaning|
|---|---|
|`-n VALUE`|niceness value, range **-20 (highest priority) to 19 (lowest priority)**|
|`-p PID`|(renice only) target an existing process by PID|

**Developer bridge:** think of niceness as inverse priority — a _higher_ number means the process is "nicer" to everyone else, i.e. it yields CPU more readily. Default niceness for any process you launch is `0`. Only `root` can set _negative_ niceness (boost priority above default) — a regular user can only make things "nicer" (lower priority), never higher, which makes sense: you shouldn't be able to starve other users' processes on a shared box.

**EXAMPLE:**

In [ ]:
$ nice -n 10 sleep 300 &
[1] 8102

$ ps -o pid,ni,cmd -p 8102
  PID  NI CMD
 8102  10 sleep 300

The `NI` column (visible in `ps -o` or `top`) shows the niceness value directly — this is the same `NI` column you already know from `top`.

Now adjust an already-running process:

In [ ]:
$ renice -n 15 -p 8102
8102 (process ID) old priority 10, new priority 15

**READ THE OUTPUT:** In `top`, sort by `%CPU` — a process with `NI` at `19` sitting at the bottom of usage even under load is correctly deprioritized. A process with `NI` at `0` (or negative) consuming everything is a candidate to `renice` down without killing it.

**TRY NOW:**

In [ ]:
dennie@HP:~/linux-tut$ nice -n 10 sleep 300 &
[1] 12867

In [ ]:
dennie@HP:~/linux-tut$ ps -o pid,ni,cmd -p $(pgrep -f "sleep 300")
    PID  NI CMD
  12867  10 sleep 300

In [ ]:
dennie@HP:~/linux-tut$ renice -n 19 -p 12867
12867 (process ID) old priority 10, new priority 19

In [ ]:
dennie@HP:~/linux-tut$ ps -o pid,ni,cmd -p 12867
    PID  NI CMD
  12867  19 sleep 300

Priority confirmed changed from `10` to `19`. Let's clean up:

In [ ]:
dennie@HP:~/linux-tut$ kill -15 12867
[1]+  Terminated              nice -n 10 sleep 300
dennie@HP:~/linux-tut$ pgrep -a sleep

Clean — confirmed. One thing worth noting in that output: the job control message says `Terminated nice -n 10 sleep 300`, even though `ps` showed the actual running command as just `sleep 300`. That's not a contradiction — `nice` executed and then **exec'd into** `sleep` (replaced its own process image with sleep's, same PID, that's why niceness carried over), so the kernel/`ps` only ever see `sleep 300`. But bash's _job table_ remembers the literal line you typed at the prompt, so it reports the original invocation. Small detail, but useful to know both views exist and why they can differ.

---

**COMMAND:** `nohup` + job control (`&`, `jobs`, `fg`, `bg`)

**PURPOSE (support):** Everything you've launched with `&` so far dies the instant your terminal/SSH session closes — the shell sends SIGHUP to all its background children on exit. That's fine for a quick test, but if you SSH into a server to kick off a long migration or backup and your laptop sleeps mid-transfer, you just killed the job. `nohup` (literally "no hang up") tells the process to ignore SIGHUP so it survives your session ending.

**SYNTAX:** `nohup command &`

**Job control companions:**

|Command|Meaning|
|---|---|
|`command &`|run in background, return prompt immediately|
|`jobs`|list background/suspended jobs in _this_ shell session|
|`fg %N`|bring job N to foreground|
|`bg %N`|resume a suspended job in the background|
|`Ctrl+Z`|suspend the current foreground job (pause, not kill)|

**Developer bridge:** `nohup` is the shell-native ancestor of what `systemd` services or `docker run -d` give you for free — a process detached from the session that spawned it. Before systemd, `nohup long_script.sh &` was literally how people ran production daemons off SSH sessions.

**EXAMPLE:**

In [ ]:
$ nohup sleep 300 &
[1] 15200
$ nohup: ignoring input and appending output to 'nohup.out'

Notice: `nohup` automatically redirects stdout/stderr to a file called `nohup.out` in your current directory, since there's no terminal for it to write to anymore once you disconnect.

In [ ]:
$ jobs
[1]+  Running                 nohup sleep 300 &

**TRY NOW**:

In [ ]:
dennie@HP:~/linux-tut$ sleep 100
^Z
[1]+  Stopped                 sleep 100

Don't background it this time — let it run in the foreground, then press **Ctrl+Z**. You'll see it suspend:

Now:

In [ ]:
dennie@HP:~/linux-tut$ jobs
[1]+  Stopped                 sleep 100

In [ ]:
dennie@HP:~/linux-tut$ bg %1
[1]+ sleep 100 &
[1]+  Done                    sleep 100

In [ ]:
dennie@HP:~/linux-tut$ jobs

In [ ]:
dennie@HP:~/linux-tut$ fg %1
bash: fg: %1: no such job

That's exactly correct behavior — not an error, just timing again (100 seconds is easy to burn while reading and typing between steps).

Walk through it: `bg %1` resumed the stopped `sleep 100` in the background — bash echoes `[1]+ sleep 100 &` to confirm. By the time you glanced at output or the next prompt drew, the 100 seconds had already elapsed, so bash immediately reported `[1]+ Done sleep 100` — the job finished and was automatically removed from the job table. That's why `jobs` came back empty right after, and why `fg %1` correctly says `no such job` — there's nothing left to bring to the foreground, it already completed. This is bash cleaning up after itself exactly as designed.

You've now seen the full lifecycle: **foreground → Ctrl+Z (stopped) → bg (running in background) → Done (completed, auto-removed)**. That's the complete mental model — solid work.

---

**TICKET SCENARIO:** "I need to run a 2-hour log-archival script on the server, but I'll lose SSH connectivity on my train commute." Diagnosis/fix: `nohup ./archive_logs.sh &`, then disconnect freely — check back later with `pgrep -af archive_logs` or `cat nohup.out` to confirm it completed. (In real production you'd more likely use `systemd` or `tmux`/`screen` for this, but `nohup` is the fundamental building block worth understanding first.)

**GOTCHA:** `nohup` alone does **not** protect against SIGKILL or a reboot — it only ignores SIGHUP. Also, people often forget the trailing `&` — `nohup command` without `&` still runs in the foreground and blocks your terminal; `nohup` and backgrounding are two separate, complementary things.

---